<a href="https://colab.research.google.com/github/emmanuelmassawe/Sentiment-Analysis-with-huggingface_transformers/blob/main/Sentiment_Analysis_with_huggingface_transformers_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [88]:
!pip install -U transformers
!pip install -U kagglehub
!pip install -U kaggle
!pip install -U evaluate


## importing necessary file

In [89]:
#importing necessary libraries
import pandas as pd
import argparse

import numpy as np
import evaluate
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

In [90]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kashishparmar02/social-media-sentiments-analysis-dataset")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'social-media-sentiments-analysis-dataset' dataset.
Path to dataset files: /kaggle/input/social-media-sentiments-analysis-dataset


In [91]:
import os

print(os.listdir(path))

['sentimentdataset.csv']


## loading the dataset

In [92]:
df =pd.read_csv(os.path.join(path,'sentimentdataset.csv'))

In [93]:
df.head(10)

,Unnamed: 0.1,Unnamed: 0,Text,Sentiment,Timestamp,User,Platform,Hashtags,Retweets,Likes,Country,Year,Month,Day,Hour
0,0,0,Enjoying a beautiful day at the park! ...,Positive,2023-01-15 12:30:00,User123,Twitter,#Nature #Park,15.0,30.0,USA,2023,1,15,12
1,1,1,Traffic was terrible this morning. ...,Negative,2023-01-15 08:45:00,CommuterX,Twitter,#Traffic #Morning,5.0,10.0,Canada,2023,1,15,8
2,2,2,Just finished an amazing workout! 💪 ...,Positive,2023-01-15 15:45:00,FitnessFan,Instagram,#Fitness #Workout,20.0,40.0,USA,2023,1,15,15
3,3,3,Excited about the upcoming weekend getaway! ...,Positive,2023-01-15 18:20:00,AdventureX,Facebook,#Travel #Adventure,8.0,15.0,UK,2023,1,15,18
4,4,4,Trying out a new recipe for dinner tonight. ...,Neutral,2023-01-15 19:55:00,ChefCook,Instagram,#Cooking #Food,12.0,25.0,Australia,2023,1,15,19
5,5,5,Feeling grateful for the little things in lif...,Positive,2023-01-16 09:10:00,GratitudeNow,Twitter,#Gratitude #PositiveVibes,25.0,50.0,India,2023,1,16,9
6,6,6,Rainy days call for cozy blankets and hot coc...,Positive,2023-01-16 14:45:00,RainyDays,Facebook,#RainyDays #Cozy,10.0,20.0,Canada,2023,1,16,14
7,7,7,The new movie release is a must-watch! ...,Positive,2023-01-16 19:30:00,MovieBuff,Instagram,#MovieNight #MustWatch,15.0,30.0,USA,2023,1,16,19
8,8,8,Political discussions heating up on the timel...,Negative,2023-01-17 08:00:00,DebateTalk,Twitter,#Politics #Debate,30.0,60.0,USA,2023,1,17,8
9,9,9,Missing summer vibes and beach days. ...,Neutral,2023-01-17 12:20:00,BeachLover,Facebook,#Summer #BeachDays,18.0,35.0,Australia,2023,1,17,12


In [94]:
df.shape

(732, 15)

## dropping unnecessary columns

In [95]:
df = df[["Text", "Sentiment"]].dropna()

print(df.head())
print(df["Sentiment"].value_counts())

                                                Text    Sentiment
0   Enjoying a beautiful day at the park!        ...   Positive  
1   Traffic was terrible this morning.           ...   Negative  
2   Just finished an amazing workout! 💪          ...   Positive  
3   Excited about the upcoming weekend getaway!  ...   Positive  
4   Trying out a new recipe for dinner tonight.  ...   Neutral   
Sentiment
Positive         44
Joy              42
Excitement       32
Neutral          14
Contentment      14
                 ..
Triumph           1
Heartwarming      1
Obstacle          1
Sympathy          1
Pressure          1
Name: count, Length: 279, dtype: int64


In [96]:
# Ondoa missing values
df = df.dropna(subset=["Text", "Sentiment"]).copy()

# Safisha labels
df["Sentiment"] = df["Sentiment"].astype(str).str.strip().str.lower()

# Baki na positive na negative
df = df[df["Sentiment"].isin(["positive", "negative"])].copy()

print(df["Sentiment"].value_counts())
print("Dataset shape:", df.shape)

Sentiment
positive    45
negative     4
Name: count, dtype: int64
Dataset shape: (49, 2)


In [97]:
print(df.columns.tolist())

['Text', 'Sentiment']


In [98]:
df['Sentiment']=df['Sentiment'].map({'positive':1,'negative':0})

In [99]:
df.head(2)

,Text,Sentiment
0,Enjoying a beautiful day at the park! ...,1
1,Traffic was terrible this morning. ...,0


In [100]:
df.head(2)

,Text,Sentiment
0,Enjoying a beautiful day at the park! ...,1
1,Traffic was terrible this morning. ...,0


## performing train_test and split

In [101]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["Sentiment"]
)

print("Train:", len(train_df))
print("Test:", len(test_df))

Train: 39
Test: 10


## Convert kwenda Hugging Face Dataset

In [102]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["Text", "Sentiment"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["Text", "Sentiment"]],
    preserve_index=False
)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['Text', 'Sentiment'],
    num_rows: 39
})
Dataset({
    features: ['Text', 'Sentiment'],
    num_rows: 10
})


## loading our model

In [103]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

In [104]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    #id2label=id2label,
    #label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [105]:
id2label = {
    0: "NEGATIVE",
    1: "POSITIVE"
}

label2id = {
    "NEGATIVE": 0,
    "POSITIVE": 1
}

model.config.id2label = id2label
model.config.label2id = label2id

In [106]:
def tokenize_function(examples):
    return tokenizer(
        examples["Text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/39 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [107]:
print(train_tokenized.column_names)
print(test_tokenized.column_names)

['Text', 'Sentiment', 'input_ids', 'token_type_ids', 'attention_mask']
['Text', 'Sentiment', 'input_ids', 'token_type_ids', 'attention_mask']


## EVALUATION METRIC

In [108]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

TRAINING CONFIGURATION

In [109]:
training_args = TrainingArguments(
    output_dir="/content/sentiment_model",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="f1",
    greater_is_better=True,

    #logging_dir="/content/logs",
    logging_steps=50,

    report_to="none"
)

In [110]:
from transformers import DataCollatorWithPadding

# Use a data collator to handle dynamic padding during batching
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

ValueError: The model did not return a loss from the inputs, only the following keys: logits. For reference, the inputs it received are input_ids,attention_mask.